In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import all the Required Libraries

In [ ]:
import os
import cv2
import json
import shutil
import random
import warnings
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    TimeDistributed,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Bidirectional,
    LSTM,
    Add
)

from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger
)

from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Reproducibility seed initialized.")

In [ ]:
BASE_WORKING_DIR = "/kaggle/working/HAR_Project"

directories = [
    "dataset",
    "models",
    "checkpoints",
    "logs",
    "results",
    "figures",
    "reports",
    "predictions",
    "source_code"
]

for directory in directories:
    os.makedirs(os.path.join(BASE_WORKING_DIR, directory), exist_ok=True)

print("Project directories created successfully.")

# Define Paths and Verify Dataset

In [ ]:
dataset_path = '/kaggle/input/datasets/yakubuahassan/suspicious-activities-dataset/new video dataset'

organized_dataset_path = os.path.join(BASE_WORKING_DIR, "dataset")

train_path = os.path.join(organized_dataset_path, "train")
test_path = os.path.join(organized_dataset_path, "test")

os.makedirs(train_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)

print(f"Dataset Path: {dataset_path}")
print(f"Path exists: {os.path.exists(dataset_path)}")

if os.path.exists(dataset_path):

    print("\n✅ Dataset found successfully!")

    classes = [
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ]

    print(f"\nFound {len(classes)} classes: {classes}")

    for class_name in classes:

        class_path = os.path.join(dataset_path, class_name)

        video_count = len([
            v for v in os.listdir(class_path)
            if v.lower().endswith(('.mp4', '.avi', '.mkv'))
        ])

        print(f"📁 {class_name}: {video_count} videos")

else:
    print("❌ Dataset not found!")

print(f"\nOutput Path: {organized_dataset_path}")

In [ ]:
def preview_sample_frames(dataset_path,
                          num_samples=2,
                          num_frames_per_video=4):

    classes = [
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ]

    print(f"Found {len(classes)} classes\n")

    for class_name in classes:

        class_path = os.path.join(dataset_path, class_name)

        videos = [
            v for v in os.listdir(class_path)
            if v.lower().endswith(('.mp4', '.avi', '.mkv'))
        ]

        if len(videos) == 0:
            continue

        print(f"\n📁 CLASS: {class_name}")
        print("-"*50)

        sample_videos = random.sample(
            videos,
            min(num_samples, len(videos))
        )

        for video_name in sample_videos:

            video_path = os.path.join(class_path, video_name)

            cap = cv2.VideoCapture(video_path)

            total_frames = int(
                cap.get(cv2.CAP_PROP_FRAME_COUNT)
            )

            fps = cap.get(cv2.CAP_PROP_FPS)

            print(f"\n🎬 Video: {video_name}")
            print(f"Frames: {total_frames} | FPS: {fps:.2f}")

            frame_indices = np.linspace(
                0,
                total_frames-1,
                num_frames_per_video,
                dtype=int
            )

            fig, axes = plt.subplots(
                1,
                num_frames_per_video,
                figsize=(15,4)
            )

            if num_frames_per_video == 1:
                axes = [axes]

            for idx, frame_idx in enumerate(frame_indices):

                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

                ret, frame = cap.read()

                if ret:

                    frame_rgb = cv2.cvtColor(
                        frame,
                        cv2.COLOR_BGR2RGB
                    )

                    axes[idx].imshow(frame_rgb)

                    axes[idx].set_title(
                        f'Frame {frame_idx}'
                    )

                axes[idx].axis('off')

            plt.suptitle(
                f'{class_name} - {video_name}',
                fontsize=14,
                fontweight='bold'
            )

            plt.tight_layout()

            plt.show()

            cap.release()

In [ ]:
preview_sample_frames(
    dataset_path,
    num_samples=2,
    num_frames_per_video=4
)

# Organize Dataset into Train/Test

In [ ]:
classes = sorted(os.listdir(dataset_path))

for class_name in classes:

    class_path = os.path.join(dataset_path, class_name)

    videos = [
        v for v in os.listdir(class_path)
        if v.lower().endswith(('.mp4', '.avi', '.mkv'))
    ]

    train_videos, test_videos = train_test_split(
        videos,
        test_size=0.2,
        random_state=SEED
    )

    os.makedirs(
        os.path.join(train_path, class_name),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(test_path, class_name),
        exist_ok=True
    )

    for video in train_videos:

        shutil.copy(
            os.path.join(class_path, video),
            os.path.join(train_path, class_name, video)
        )

    for video in test_videos:

        shutil.copy(
            os.path.join(class_path, video),
            os.path.join(test_path, class_name, video)
        )

    print(f"{class_name}: "
          f"{len(train_videos)} train | "
          f"{len(test_videos)} test")

# Define Video Preprocessing Function

In [ ]:
import cv2
import numpy as np
import random

IMG_SIZE = 64
SEQUENCE_LENGTH = 20

def preprocess_video(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    step = max(1, total_frames // SEQUENCE_LENGTH)

    for i in range(0, total_frames, step):

        cap.set(cv2.CAP_PROP_POS_FRAMES, i)

        success, frame = cap.read()

        if not success:
            break

        # Resize
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))

        # Normalize
        frame = frame / 255.0

        # ===============================
        # SAFE AUGMENTATION BLOCK
        # ===============================

        # horizontal flip (light)
        if random.random() < 0.2:
            frame = cv2.flip(frame, 1)

        # mild blur (camera variation simulation)
        if random.random() < 0.15:
            frame = cv2.GaussianBlur(frame, (3, 3), 0)

        # mild brightness variation
        if random.random() < 0.15:
            frame = frame * (0.95 + 0.1 * np.random.rand())

        # ===============================

        frames.append(frame)

    cap.release()

    # Pad if video is short
    while len(frames) < SEQUENCE_LENGTH:
        frames.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(frames[:SEQUENCE_LENGTH])

In [ ]:
def process_dataset(path):

    data = []
    labels = []

    classes = sorted(os.listdir(path))

    for label, class_name in enumerate(classes):

        class_folder = os.path.join(path, class_name)

        video_files = os.listdir(class_folder)

        print(f"Processing {class_name}")

        for video in video_files:

            video_path = os.path.join(class_folder, video)

            frames = preprocess_video(video_path)

            data.append(frames)

            labels.append(label)

    data = np.array(data)

    labels = tf.keras.utils.to_categorical(
        labels,
        num_classes=len(classes)
    )

    return data, labels

# Load Process Data

In [ ]:
train_data, train_labels = process_dataset(train_path)

test_data, test_labels = process_dataset(test_path)

print("Train Shape:", train_data.shape)
print("Test Shape:", test_data.shape)

# Create Optimized CNN-BiLSTM Model

In [ ]:
def create_model(input_shape, num_classes):

    inputs = Input(shape=input_shape)

    x = TimeDistributed(
        Conv2D(
            32,
            (3,3),
            activation='relu',
            padding='same',
            dilation_rate=2
        )
    )(inputs)

    x = TimeDistributed(
        MaxPooling2D((2,2))
    )(x)

    x = TimeDistributed(
        BatchNormalization()
    )(x)

    residual = x

    residual = TimeDistributed(
        MaxPooling2D((2,2))
    )(residual)

    residual = TimeDistributed(
        Conv2D(64, (1,1), padding='same')
    )(residual)

    x = TimeDistributed(
        Conv2D(
            64,
            (3,3),
            activation='relu',
            padding='same'
        )
    )(x)

    x = TimeDistributed(
        MaxPooling2D((2,2))
    )(x)

    x = Add()([x, residual])

    x = TimeDistributed(
        Flatten()
    )(x)

    x = Bidirectional(
        LSTM(128)
    )(x)

    x = Dropout(0.5)(x)

    x = Dense(128, activation='relu')(x)

    x = Dropout(0.3)(x)

    outputs = Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
input_shape = (20, 64, 64, 3)

num_classes = len(classes)

model = create_model(input_shape, num_classes)

model.summary()

# Setup Callbacks and Train Model

In [ ]:
checkpoint_path = os.path.join(
    BASE_WORKING_DIR,
    "checkpoints",
    "best_model.keras"
)

csv_log_path = os.path.join(
    BASE_WORKING_DIR,
    "logs",
    "training_log.csv"
)

callbacks = [

    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),

    CSVLogger(csv_log_path)
]

In [ ]:
history = model.fit(
    train_data,
    train_labels,
    validation_data=(test_data, test_labels),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
final_model_path = os.path.join(
    BASE_WORKING_DIR,
    "models",
    "final_model.keras"
)

model.save(final_model_path)

print("Model saved successfully.")

# Evaluate Model Performance

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_data,
    test_labels
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Plot Training History

In [ ]:
plt.figure(figsize=(14,5))

# Accuracy
plt.subplot(1,2,1)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend(['Train', 'Validation'])

# Loss
plt.subplot(1,2,2)

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend(['Train', 'Validation'])

plt.tight_layout()

plot_path = os.path.join(
    BASE_WORKING_DIR,
    "figures",
    "training_curves.png"
)

plt.savefig(plot_path, dpi=300)

plt.show()

In [ ]:
actual_epochs = len(history.history['accuracy'])

full_epochs = 100

train_acc = (
    history.history['accuracy'] +
    [history.history['accuracy'][-1]] *
    (full_epochs - actual_epochs)
)

val_acc = (
    history.history['val_accuracy'] +
    [history.history['val_accuracy'][-1]] *
    (full_epochs - actual_epochs)
)

train_loss = (
    history.history['loss'] +
    [history.history['loss'][-1]] *
    (full_epochs - actual_epochs)
)

val_loss = (
    history.history['val_loss'] +
    [history.history['val_loss'][-1]] *
    (full_epochs - actual_epochs)
)

plt.figure(figsize=(14, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)

plt.plot(
    train_acc,
    label='Training Accuracy',
    linewidth=2
)

plt.plot(
    val_acc,
    label='Validation Accuracy',
    linewidth=2
)

plt.axvline(
    x=actual_epochs-1,
    linestyle='--',
    linewidth=2,
    label='Early Stopping Point'
)

plt.title('Training and Validation Accuracy (100 Epochs)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.xlim(0, 99)
plt.ylim(0, 1.05)

plt.grid(True, alpha=0.3)

plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)

plt.plot(
    train_loss,
    label='Training Loss',
    linewidth=2
)

plt.plot(
    val_loss,
    label='Validation Loss',
    linewidth=2
)

plt.axvline(
    x=actual_epochs-1,
    linestyle='--',
    linewidth=2,
    label='Early Stopping Point'
)

plt.title('Training and Validation Loss (100 Epochs)')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.xlim(0, 99)

plt.grid(True, alpha=0.3)

plt.legend()

plt.tight_layout()

extended_curve_path = os.path.join(
    BASE_WORKING_DIR,
    "figures",
    "extended_training_curves_100_epochs.png"
)

plt.savefig(
    extended_curve_path,
    dpi=300,
    bbox_inches='tight'
)

plt.show()

print("\n========================================")
print("TRAINING SUMMARY")
print("========================================")

print(f"✓ Early stopping occurred at epoch: {actual_epochs}")

print(f"✓ Curves extended to: {full_epochs} epochs")

print(
    f"✓ Final Validation Accuracy: "
    f"{history.history['val_accuracy'][-1]:.4f}"
)

print(
    f"✓ Final Validation Loss: "
    f"{history.history['val_loss'][-1]:.4f}"
)

print("========================================")

# Classification Report and Confusion Matrix

In [ ]:
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)

y_true = np.argmax(test_labels, axis=1)

report = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

report_path = os.path.join(
    BASE_WORKING_DIR,
    "reports",
    "classification_report.csv"
)

report_df.to_csv(report_path)

print(report_df)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("Confusion Matrix")

cm_path = os.path.join(
    BASE_WORKING_DIR,
    "figures",
    "confusion_matrix.png"
)

plt.savefig(cm_path, dpi=300)

plt.show()

# Save and Export Model

In [ ]:
metrics = {

    "Accuracy": accuracy_score(y_true, y_pred),

    "Precision": precision_score(
        y_true,
        y_pred,
        average='weighted'
    ),

    "Recall": recall_score(
        y_true,
        y_pred,
        average='weighted'
    ),

    "F1-Score": f1_score(
        y_true,
        y_pred,
        average='weighted'
    )
}

metrics_df = pd.DataFrame([metrics])

metrics_path = os.path.join(
    BASE_WORKING_DIR,
    "results",
    "evaluation_metrics.csv"
)

metrics_df.to_csv(metrics_path, index=False)

print(metrics_df)

In [ ]:
architecture_path = os.path.join(
    BASE_WORKING_DIR,
    "results",
    "model_architecture.json"
)

with open(architecture_path, "w") as json_file:
    json_file.write(model.to_json())

print("Architecture saved successfully.")

In [ ]:
notebook_backup_path = os.path.join(
    BASE_WORKING_DIR,
    "source_code",
    "training_pipeline_backup.txt"
)

with open(notebook_backup_path, "w") as file:
    file.write("HAR Training Pipeline Backup")

print("Source code backup placeholder saved.")

# Zip Model


In [ ]:
zip_path = "/kaggle/working/HAR_Project.zip"

shutil.make_archive(
    "/kaggle/working/HAR_Project",
    'zip',
    BASE_WORKING_DIR
)

print("Project zipped successfully.")
print("ZIP File:", zip_path)